## Chapter 5. 결정 트리 (Decision Trees)

### 1. 결정 트리의 기본 개념

결정 트리는 데이터를 하나씩 조건으로 쪼개 나가면서 최종 예측에 도달하는 지도학습 알고리즘이다. 분류든 회귀든 다 쓸 수 있고, 특성들 사이의 복잡한 비선형 관계도 곡선을 직접 가정하지 않고 표현해낼 수 있다는 게 특징이다. 그리고 무엇보다 다음 장에서 배우는 랜덤 포레스트를 비롯한 여러 앙상블 기법들이 전부 결정 트리를 기본 구성 요소로 삼기 때문에, 이 장의 내용을 제대로 이해하고 넘어가는 게 이후 챕터를 소화하는 데도 중요하다.

트리는 맨 위 루트 노드에서 출발한다. 각 분할 노드는 "이 특성 값이 어떤 기준보다 작은가?" 같은 질문 하나로 데이터를 두 갈래로 나누고, 이 과정을 계속 반복하다가 더 이상 쪼갤 필요가 없다고 판단되는 리프 노드에 도달하면 거기서 예측값을 내놓는다.

붓꽃 분류를 예로 들면, 먼저 꽃잎 길이를 확인해서 큰 갈래로 나누고, 그래도 애매한 경우엔 꽃잎 너비를 한 번 더 확인해서 품종을 정하는 식이다. 이렇게 예측 과정 전체를 "만약 A이면 B, 아니면 C"라는 조건문 나열로 그대로 옮길 수 있다는 점 때문에, 트리 깊이가 얕을 때는 사람이 그 판단 근거를 눈으로 따라갈 수 있다. 책에서는 이런 성질을 가진 모델을 **화이트박스 모델**이라 부르고, 반대로 신경망처럼 왜 그런 예측이 나왔는지 설명하기 까다로운 모델을 **블랙박스 모델**이라 구분한다. 다만 이 "설명 가능성"은 트리가 얕을 때 얘기고, 깊이가 깊어지고 노드 수가 수백 개로 늘어나면 이론적으로는 여전히 조건문 나열이지만 사람이 실제로 따라가며 이해하기는 어려워진다는 점은 짚고 넘어갈 만하다.

또 하나 실용적으로 중요한 특징은, 결정 트리가 특성의 스케일을 맞추거나 평균을 빼는 전처리를 거의 요구하지 않는다는 점이다. 로지스틱 회귀나 SVM처럼 거리·내적을 계산하는 모델은 특성 단위가 다르면 왜곡이 생기지만, 결정 트리는 각 특성을 "이 값이 임계값보다 큰가 작은가"로만 비교하기 때문에 특성을 몇 배로 늘리거나 줄여도 분할 결과 자체는 바뀌지 않는다.

### 2. 불순도와 클래스 확률

분류 트리는 한 노드 안에 여러 클래스가 얼마나 뒤섞여 있는지를 나타내는 지표로 **불순도**를 쓴다. 한 클래스만 들어 있는 노드는 불순도가 0인 순수 노드다.

대표적인 지표는 두 가지다.

- **지니 불순도**: 노드 안에서 클래스 $k$의 비율을 $p_k$라 할 때,

$$G = 1 - \sum_k p_k^2$$

- **엔트로피**: 정보 이론에서 가져온 개념으로,

$$H = -\sum_{k,\,p_k \neq 0} p_k \log_2(p_k)$$

한 클래스만 있으면 0이고, 클래스들이 고르게 섞일수록 커진다.

두 지표는 실전에서 만들어내는 트리가 대체로 비슷하다. 차이가 있다면, 지니 불순도는 로그 계산이 없어 조금 더 빠르고(그래서 사이킷런의 기본값이기도 하다), 엔트로피는 특정 상황에서 지니보다 약간 더 균형 잡힌 트리를 만드는 경향이 있다고 알려져 있다 — 지니는 가장 빈도가 높은 클래스 하나를 가지에서 고립시키는 쪽으로, 엔트로피는 조금 더 균형 있게 나누는 쪽으로 기운다는 관찰이다. 다만 이 차이가 실제 성능에 미치는 영향은 크지 않은 경우가 많아서, 계산이 간단한 지니를 기본으로 쓰고 특별히 다른 결과가 필요할 때만 엔트로피로 바꿔 비교해보는 정도로 접근하면 된다.

결정 트리는 클래스 하나를 딱 찍어주는 것 말고도 클래스별 확률을 추정할 수 있다. 방법은 단순한데, 입력이 도달한 리프 노드 안에서 각 클래스가 차지하는 비율을 그대로 확률로 쓴다. 예를 들어 어떤 리프에 훈련 샘플이 A 클래스 9개, B 클래스 1개 있었다면, 그 리프에 도달하는 모든 새 샘플에 대해 A일 확률 90%, B일 확률 10%라고 답하는 식이다.

여기서 꼭 기억해야 할 한계가 있다. **같은 리프에 도달한 샘플은 전부 똑같은 확률을 받는다.** 리프 안에서 그 샘플이 A 쪽 경계에 더 가까운지, B 쪽 경계에 더 가까운지는 전혀 구분하지 않는다는 뜻이다. 즉 이 확률은 어디까지나 "이 리프에 있던 훈련 데이터의 클래스 구성 비율"일 뿐, 개별 샘플의 미세한 위치 차이를 반영한 진짜 확률적 추정치는 아니다. 이 점은 나중에 소프트 보팅이나 그레이디언트 부스팅에서 확률 출력의 질을 논할 때 다시 떠오르는 문제이기도 하다.

### 3. CART 알고리즘의 학습 과정

사이킷런은 결정 트리를 학습시킬 때 **CART(Classification and Regression Tree)** 알고리즘을 쓴다. CART는 항상 각 노드를 정확히 두 개의 자식 노드로 나누는 이진 트리만 만든다(ID3처럼 한 노드에서 여러 갈래로 나누는 방식이 아니다).

학습 절차를 순서대로 풀면 다음과 같다.

1. 현재 노드에서 사용할 수 있는 특성 하나와 그 특성의 분할 임계값 하나의 조합을 후보로 삼는다.
2. 그 조합으로 나눴을 때 두 자식 노드의 불순도가 가장 작아지는 조합을 고른다.
3. 각 자식 노드에서 같은 과정을 재귀적으로 반복한다.
4. 최대 깊이에 도달했거나, 더 이상 불순도를 줄이는 유효한 분할이 없거나, 사용자가 지정한 정지 조건(뒤에서 다룰 `min_samples_split` 등)에 걸리면 멈춘다.

여기서 헷갈리기 쉬운 부분이 하나 있는데, 두 자식 노드의 불순도를 단순 평균하는 게 아니라 **각 노드에 속한 샘플 수를 반영한 가중평균**을 최소화한다는 점이다.

$$J(k, t_k) = \frac{m_{\text{left}}}{m} G_{\text{left}} + \frac{m_{\text{right}}}{m} G_{\text{right}}$$

$m_{\text{left}}, m_{\text{right}}$는 왼쪽·오른쪽 자식 노드의 샘플 수, $m$은 현재 노드 전체 샘플 수다. 이렇게 가중치를 두는 이유는 직관적이다 — 샘플 10개 중 1개짜리 아주 작은 그룹만 완벽하게 순수하게 만드는 분할보다는, 전체 샘플 대부분을 잘 구분해내는 분할이 실제로 더 유용하기 때문이다. 가중치 없이 단순 평균만 썼다면, 크기가 극단적으로 작은 자식 노드 하나를 순수하게 만드는 방향으로 알고리즘이 편향될 위험이 있다.

CART는 딱 지금 이 노드에서 가장 좋아 보이는 분할을 고르고 넘어가는 **탐욕(greedy) 알고리즘**이다. 이번 분할이 앞으로 몇 단계 뒤에 어떤 결과를 낳을지 미리 내다보고 최적화하지 않는다. 그래서 CART가 만든 트리가 전체적으로 가장 좋은 트리라는 보장은 없다 — 이론적으로 최적의 트리를 찾는 문제는 NP-완전 문제로 알려져 있어서, 데이터가 조금만 커져도 가능한 모든 트리 구조를 다 비교해보는 건 현실적으로 불가능하다. 그래서 "완벽하지는 않아도 합리적인 시간 안에 꽤 괜찮은 트리"를 얻는 실용적 타협으로 탐욕 방식을 쓰는 것이다.

계산 복잡도 면에서 보면, 트리가 대략 균형 잡혀 있다면 예측할 때는 루트에서 리프까지 한 번만 내려가면 되므로 노드 수는 트리 깊이에 비례하고, 균형 잡힌 이진 트리의 깊이는 대략 $\log_2 m$ 수준이다. 그래서 예측 복잡도는 특성 수와 무관하게 $O(\log m)$ 정도로 매우 빠르다. 반면 훈련 과정에서는 각 노드마다 모든 특성 $n$개에 대해 데이터를 정렬해서 최적의 분할점을 찾아야 하므로, 전체 훈련 복잡도는 대략 $O(n \cdot m \log m)$ 수준으로 예측보다 훨씬 무겁다.

### 4. 과적합과 규제

결정 트리에 아무런 제약을 두지 않으면, 훈련 데이터를 거의 완벽하게 설명할 때까지 계속 잘게 쪼갠다. 문제는 이 과정에서 일반적인 패턴뿐 아니라 훈련 데이터에만 있는 우연한 잡음까지 학습해버린다는 것이다. 이런 트리는 훈련 정확도는 매우 높지만 새로운 데이터에서는 성능이 뚝 떨어지는, 전형적인 과적합 상태가 된다.

여기서 "결정 트리는 비모수 모델이다"라는 말을 오해하기 쉬운데, 파라미터가 아예 없다는 뜻이 아니다. 오히려 각 노드의 분할 기준(어떤 특성, 어떤 임계값)이 전부 모델의 파라미터라고 볼 수 있고, 그 수가 매우 많을 수도 있다. "비모수적"이라는 말의 정확한 의미는, 선형 회귀처럼 학습을 시작하기 전에 파라미터 개수가 미리 고정되어 있는 게 아니라, 데이터가 얼마나 복잡한지에 따라 트리의 구조와 파라미터 수가 훈련 중에 자유롭게 늘어날 수 있다는 뜻이다. 그래서 규제를 걸어주지 않으면 트리가 데이터에 맞춰 한없이 복잡해질 수 있는 것이다.

과적합을 억제하는 주요 하이퍼파라미터는 다음과 같다.

- `max_depth`: 트리가 내려갈 수 있는 최대 깊이. 작게 잡을수록 복잡한 분할이 제한된다.
- `max_leaf_nodes`: 리프 노드 수의 상한.
- `min_samples_split`: 노드를 더 쪼개려면 최소한 이만큼의 샘플이 있어야 한다는 조건.
- `min_samples_leaf`: 하나의 리프가 최소한 가져야 할 샘플 수.
- `min_weight_fraction_leaf`: `min_samples_leaf`와 비슷하지만, 샘플 개수 대신 전체 샘플 가중치 중 비율로 지정한다. 샘플마다 가중치가 다르게 주어지는 상황(예: 클래스 불균형 보정)에서 유용하다.
- `max_features`: 각 노드에서 분할 후보로 검토할 특성 수. 다음 장 랜덤 포레스트에서 더 중요해지는 파라미터다.
- `min_impurity_decrease`: 분할이 불순도를 이 값 이상 줄이지 못하면 그 분할을 허용하지 않는다.
- `ccp_alpha`: 비용 복잡도 가지치기(cost-complexity pruning)의 강도. 값이 클수록 더 많은 가지가 잘려나간다.

대체로 `max_*` 계열 값을 줄이거나 `min_*` 계열 값을 늘리면 모델의 자유도가 줄어들어 규제가 강해진다. 다만 이 규제를 너무 세게 걸면 이번엔 반대로 데이터에 실제로 존재하는 유의미한 패턴조차 잡아내지 못하는 **과소적합**이 생긴다는 점을 항상 염두에 둬야 한다.

이 중에서도 특히 **`max_depth`와 `min_samples_leaf`**가 실무에서 가장 자주 조정되는 파라미터다. 깊이를 제한하면 트리가 통째로 지나치게 복잡해지는 걸 막을 수 있고, 리프의 최소 샘플 수를 늘리면 몇 개 안 되는 샘플, 혹은 잡음에 가까운 이상치 하나 때문에 생기는 불필요한 분할을 줄일 수 있다. 실전에서는 교차검증으로 이 값들을 그리드 서치하며 최적값을 찾는 경우가 많다.

가지치기(pruning)는 애초에 제약을 걸고 트리를 키우는 대신, 일단 크게 키운 트리에서 불필요한 가지를 나중에 잘라내는 접근이다. 책에서는 두 가지 아이디어를 소개하는데, 하나는 비용 복잡도(트리 크기와 오차를 함께 고려한 비용 함수)를 기준으로 가지를 잘라내는 방식 — 사이킷런의 `ccp_alpha`가 여기 해당한다 — 이고, 다른 하나는 어떤 분할이 만들어낸 불순도 개선이 통계적으로 우연이 아니라고 볼 만큼 충분히 유의미한지($\chi^2$ 검정 등으로) 판단해서, 유의미하지 않은 분할이면 아예 제거하는 접근이다. 다만 후자는 개념적으로 소개되는 것이고, 사이킷런이 기본으로 제공하는 방식은 전자인 비용 복잡도 가지치기다.

### 5. 회귀 문제에서의 결정 트리

회귀 트리는 클래스를 고르는 대신 연속적인 값을 예측한다. 기본적인 제곱오차 기준을 쓸 때, 각 리프는 그 리프에 속한 훈련 샘플들의 타깃값 평균을 예측값으로 내놓는다.

분할 기준도 자연스럽게 바뀐다. 분류 트리가 불순도(지니·엔트로피)를 최소화하는 분할을 찾았다면, 회귀 트리는 **평균제곱오차(MSE)**를 최소화하는 분할을 찾는다.

$$J(k, t_k) = \frac{m_{\text{left}}}{m} \text{MSE}_{\text{left}} + \frac{m_{\text{right}}}{m} \text{MSE}_{\text{right}}, \qquad \text{MSE}_{\text{node}} = \frac{1}{m_{\text{node}}}\sum_{i \in \text{node}} (\hat{y}_{\text{node}} - y^{(i)})^2$$

분류 트리의 가중평균 불순도 공식과 형태가 완전히 똑같다는 점이 흥미로운데, 결국 "어느 쪽으로 나눠야 각 그룹 안이 더 동질적이 되는가"라는 CART의 기본 아이디어는 분류든 회귀든 동일하고, 다만 "동질적"을 측정하는 지표만 클래스 비율 기반에서 값의 분산 기반으로 바뀌는 것이다.

같은 리프에 속한 모든 샘플에는 똑같은 예측값(그 리프의 평균)이 나가기 때문에, 예측 함수를 그래프로 그리면 매끄러운 곡선이 아니라 계단 모양이 된다. 깊이를 늘리면 계단 폭이 좁아지면서 더 세밀한 관계를 표현할 수 있지만, 분류 트리와 마찬가지로 너무 깊어지면 각 계단이 개별 샘플 하나하나에 맞춰지는 과적합으로 이어진다.

### 6. 결정 트리의 장점과 한계

결정 트리는 사용하기 쉽고, 트리가 작을 때는 예측 근거를 사람이 직접 따라가며 설명할 수 있다. 특성 스케일링 같은 전처리 부담이 적고, 분류와 회귀 양쪽에 다 쓸 수 있다는 것도 실용적인 장점이다.

반면 구조적인 한계가 두 가지 있다.

첫째, **좌표축 방향에 민감하다.** 한 번의 분할에서 단 하나의 특성만 기준으로 삼기 때문에, 결정 경계는 항상 좌표축에 평행한 직선(또는 초평면)들의 조합으로만 만들어진다. 만약 진짜 결정 경계가 대각선 형태라면, 트리는 그 대각선을 계단식으로 여러 번 잘게 쪼개서 근사해야 한다. 같은 데이터라도 45도쯤 회전시키면 원래는 분할 한 번으로 끝났을 경계가 훨씬 여러 번의 분할을 필요로 하게 될 수 있다는 뜻이다. 책에서는 이 문제를 완화하는 방법으로, 데이터를 스케일링한 뒤 PCA로 회전시켜 분산이 큰 방향을 좌표축에 맞춰주는 방법을 언급한다. 다만 이건 만능 해법은 아니고, 데이터 구조에 따라 오히려 도움이 안 될 수도 있다.

둘째, **분산이 높다.** 훈련 데이터를 살짝 바꾸거나(예: 샘플 몇 개를 빼거나), 같은 데이터라도 랜덤성이 개입하는 설정(동점 처리 등)이 조금만 달라져도 트리의 구조 자체가 크게 흔들릴 수 있다. 심지어 훈련 데이터에서 가장 위쪽 루트 노드의 분할 특성이 통째로 바뀌는 경우도 드물지 않다. `random_state`를 고정하는 건 같은 코드를 다시 실행했을 때 똑같은 결과를 재현하는 데는 도움이 되지만, "데이터가 바뀌면 트리도 크게 바뀐다"는 근본적인 불안정성 자체를 해결해주지는 않는다는 점을 헷갈리지 말아야 한다.

이 높은 분산 문제를 정면으로 해결하는 방법이 바로 다음 장의 핵심 주제인, 여러 트리의 예측을 결합하는 **랜덤 포레스트**다.

## Chapter 6. 앙상블 학습과 랜덤 포레스트

### 1. 앙상블 학습의 기본 개념

앙상블 학습은 여러 모델의 예측을 한데 모아 더 나은 예측을 만들어내는 방법이다. 개별 모델 하나하나는 완벽하지 않더라도, 서로 다른 모델이 서로 다른 실수를 한다면 그 실수들이 평균 과정에서 상쇄되면서 전체적으로는 더 안정적이고 정확한 예측이 나올 수 있다.

이 아이디어를 직관적으로 뒷받침하는 게 **콩도르세의 배심원 정리(Condorcet's jury theorem)**다. 개별 분류기가 무작위 추측보다 조금이라도 나은 확률로 정답을 맞힌다면, 그리고 각 분류기의 오류가 서로 독립적이라면, 이런 분류기를 충분히 많이 모아 다수결로 결정할 때 전체 정확도는 개별 분류기 정확도보다 높아지고 분류기 수를 늘릴수록 1에 가까워진다는 것이다. 다만 이 정리가 실전에서 그대로 성립하려면 "오류가 서로 독립적"이어야 하는데, 현실에서는 같은 데이터로 비슷한 알고리즘을 학습시키면 모델들이 비슷한 실수를 반복하는 경우가 많다. 그래서 앙상블에서 중요한 건 단순히 모델 개수가 아니라 **다양성**이다. 모든 모델이 같은 샘플에서 같은 방향으로 틀린다면, 아무리 많이 모아봐야 결합 효과는 미미하다. 서로 다른 알고리즘을 섞어 쓰거나, 훈련 데이터·특성의 일부를 모델마다 다르게 주는 방식으로 다양성을 확보할 수 있다.

물론 앙상블이 공짜는 아니다. 성능이 좋아질 가능성이 큰 대신, 모델을 여러 개 유지해야 하니 계산량과 메모리 사용량이 늘어나고, 예측이 어떻게 나왔는지 해석하기도, 배포된 시스템을 운영·관리하기도 단일 모델보다 번거로워진다.

### 2. 투표 분류기

투표 분류기(Voting Classifier)는 서로 다른 여러 분류기의 결과를 모아 최종 클래스를 정하는 가장 단순한 형태의 앙상블이다.

**하드 투표**는 각 모델이 내놓은 예측 클래스 중에서 가장 많은 표를 받은 클래스를 최종 결과로 채택하는 방식이다. 다수결과 똑같다고 생각하면 된다.

**소프트 투표**는 한 단계 더 나아가서, 각 모델이 출력한 클래스별 확률을 평균 낸 뒤 그 평균 확률이 가장 높은 클래스를 선택한다. 단순히 "몇 표를 받았나"가 아니라 "얼마나 확신을 갖고 그 클래스를 예측했나"까지 반영하기 때문에, 보통 하드 투표보다 성능이 좋은 경우가 많다.

다만 소프트 투표를 쓰려면 전제 조건이 있다. 모든 기반 모델이 클래스 확률(`predict_proba`)을 출력할 수 있어야 하고, 그 확률이 실제 정답 가능성과 잘 들어맞도록 어느 정도 보정(calibration)되어 있어야 한다. 만약 어떤 모델이 실제로는 애매한 경우에도 항상 99% 확신한다는 식으로 확률을 지나치게 과신해서 출력한다면, 평균을 낼 때 그 모델의 목소리가 부당하게 커져서 오히려 전체 결합 결과를 왜곡시킬 수 있다.

사이킷런에서는 `VotingClassifier`를 쓰고, `voting` 파라미터로 하드/소프트를 정하며, `weights` 파라미터로 모델별 기여도(가중치)를 다르게 줄 수 있다.

### 3. 배깅과 페이스팅

배깅과 페이스팅은 서로 다른 알고리즘을 섞는 대신, **같은 알고리즘을 서로 다른 훈련 데이터 부분집합에** 학습시켜서 다양성을 만드는 방법이다.

- **배깅(Bagging, Bootstrap Aggregating)**: 복원추출로 훈련 부분집합을 뽑는다. 같은 샘플이 한 모델의 훈련 데이터 안에 여러 번 들어갈 수 있다.
- **페이스팅(Pasting)**: 비복원추출로 뽑는다. 한 모델의 훈련 데이터 안에서는 같은 샘플이 중복되지 않는다.

두 방식 모두 서로 다른 모델들 사이에서는 훈련 데이터가 얼마든지 겹칠 수 있다는 점은 같다. 차이는 "한 모델 안에서" 중복을 허용하느냐일 뿐이다.

학습이 끝나면, 분류에서는 (하드/소프트) 투표를, 회귀에서는 단순히 모든 모델 예측값의 평균을 최종 결과로 쓴다. 사이킷런의 `BaggingClassifier`는 기반 모델이 확률을 낼 수 있다면 자동으로 소프트 투표(확률 평균) 방식을 쓴다.

**배깅의 핵심 효과는 분산 감소다.** 결정 트리처럼 훈련 데이터가 조금만 달라져도 예측이 크게 흔들리는(즉 분산이 높은) 모델을, 서로 다른 부트스트랩 표본으로 여러 개 학습시켜 평균 내면, 개별 트리의 들쭉날쭉한 변동이 서로 상쇄되면서 전체 앙상블의 예측은 훨씬 매끄럽고 안정적으로 바뀐다. 그래서 배깅은 결정 트리처럼 분산이 크고 과적합하기 쉬운 모델에 특히 잘 맞는다. 참고로 배깅은 복원추출 과정에서 원본 데이터의 일부를 못 보게 되는 대신 부트스트랩 표본 자체의 다양성이 커서 편향은 살짝 늘고 분산은 크게 줄어드는 경향이 있고, 페이스팅은 그 반대에 가까워서, 실무에서는 대체로 배깅이 더 널리 쓰인다.

각 모델을 서로 완전히 독립적으로(서로의 결과를 기다릴 필요 없이) 학습시킬 수 있다는 것도 큰 장점이라, CPU 코어나 서버 여러 대에 나눠서 병렬로 훈련시키기 쉽다. 주요 설정은 모델 개수를 정하는 `n_estimators`, 모델 하나당 뽑을 샘플 수를 정하는 `max_samples`, 복원추출 여부를 정하는 `bootstrap`이다.

### 4. OOB 평가와 특성 샘플링

배깅에서 복원추출을 하다 보면, 어떤 샘플은 한 모델 안에서 여러 번 뽑히고, 어떤 샘플은 아예 한 번도 뽑히지 않는 일이 생긴다. 특정 모델의 훈련에 전혀 쓰이지 않은 샘플을 그 모델의 **OOB(Out-of-Bag) 샘플**이라고 부른다.

원본 훈련 세트와 같은 횟수만큼 복원추출한다고 하면, 샘플 수가 충분히 클 때 한 모델이 보게 되는 서로 다른 원본 샘플의 비율은 대략 $1 - (1 - 1/m)^m \approx 1 - e^{-1} \approx 63\%$로 수렴한다. 즉 나머지 약 37%가 그 모델에겐 한 번도 보지 못한 OOB 샘플이 되는 셈이다.

여기서 나오는 아이디어가 꽤 영리한데, 각 훈련 샘플에 대해 **그 샘플을 훈련에 쓰지 않은 모델들만 모아서** 예측을 시켜보면, 마치 그 모델들 입장에서는 처음 보는 데이터를 맞히는 것과 같은 효과가 난다. 이걸 훈련 세트의 모든 샘플에 대해 반복하면, 별도로 검증 세트를 떼어내지 않고도 앙상블의 일반화 성능을 추정할 수 있다. 다만 OOB 점수가 독립적인 테스트 세트로 측정한 점수와 정확히 일치하는 건 아니고, 어디까지나 근사적인 추정치라는 점은 감안해야 한다.

사이킷런에서는 `oob_score=True`로 이 평가를 요청하고, 학습이 끝난 뒤 `oob_score_` 속성으로 결과를 확인할 수 있다.

샘플뿐 아니라 특성도 일부만 무작위로 뽑아서 다양성을 더 키울 수 있다. 샘플과 특성을 동시에 추출하는 방식을 **랜덤 패치(Random Patches)**, 샘플은 전부 쓰되 특성만 무작위로 추출하는 방식을 **랜덤 서브스페이스(Random Subspaces)**라고 부른다. 특히 특성 수가 아주 많은 고차원 데이터에서 계산 부담을 줄이면서 동시에 모델 간 다양성도 높이는 데 유용하다.

### 5. 랜덤 포레스트와 Extra-Trees

**랜덤 포레스트**는 여러 결정 트리를 배깅 방식으로 결합한 앙상블이다. 기본적으로 각 트리는 부트스트랩 샘플로 학습되고, 여기에 한 가지가 더해진다 — **각 노드에서 분할 후보로 검토할 특성 자체도 무작위로 일부만 제한한다**는 점이다.

이게 왜 중요한지 살펴볼 필요가 있다. 만약 모든 노드에서 항상 모든 특성을 다 검토한다면, 데이터 안에 특별히 강력한(예측력이 높은) 특성이 하나 있을 경우 거의 모든 트리가 그 특성을 루트 근처에서 반복적으로 쓰게 되고, 그러면 트리들끼리 구조가 서로 비슷해져 버린다. 트리들이 비슷하면 그 예측 오차들도 서로 비슷한 방향으로 치우치기 쉬워서, 아무리 평균을 내도 분산이 잘 안 줄어든다. 반면 매 노드마다 후보 특성을 무작위로 일부만 남기면, 강력한 특성이 항상 뽑히는 것도 아니게 되면서 트리마다 서로 다른 구조가 만들어지고, 트리들 사이의 상관관계가 줄어들면서 평균을 냈을 때의 분산 감소 효과가 훨씬 커진다.

주요 하이퍼파라미터는 다음과 같다.

- `n_estimators`: 트리 개수.
- `max_features`: 매 노드에서 검토할 특성 수. 사이킷런 기본값은 분류에서는 $\sqrt{n}$, 회귀에서는 전체 특성 수를 쓰는 식으로 다르게 설정되어 있으니, 튜닝할 때 이 기본값을 기준점으로 삼으면 편하다.
- `max_depth`, `max_leaf_nodes`: 개별 트리의 복잡도를 제한.
- `min_samples_leaf`: 지나치게 작은 리프가 만들어지는 것을 방지.

**Extra-Trees(Extremely Randomized Trees)**는 여기에 무작위성을 한 겹 더 얹는다. 후보 특성을 무작위로 고르는 것뿐 아니라, 그 특성 안에서 최적의 분할 임계값을 정밀하게 탐색하는 대신 **후보 임계값 자체도 무작위로 몇 개 생성한 뒤 그중 가장 나은 것을 고른다.** 임계값을 하나하나 정밀 탐색하는 비용을 줄일 수 있어 일반 랜덤 포레스트보다 훈련 속도가 빠르고, 무작위성이 더해진 만큼 분산도 더 낮아지는 경향이 있다. 대신 각 트리 하나하나의 정확도(편향)는 살짝 나빠질 수 있는 트레이드오프가 있다. 참고로 사이킷런 구현에서 Extra-Trees의 기본값은 `bootstrap=False`로, 일반 랜덤 포레스트와 다르다는 점도 기억해둘 만하다 — 즉 기본 설정으로는 샘플 복원추출 없이 전체 훈련 세트를 그대로 쓰고, 대신 임계값 무작위화로 다양성을 확보하는 구조다.

랜덤 포레스트는 `feature_importances_` 속성을 통해 특성 중요도를 제공한다. 이 값은 숲을 이루는 모든 트리에서, 어떤 특성이 등장한 분할이 평균적으로 불순도를 얼마나 많이 줄였는지를 집계해서 정규화한 상대적 지표다. 어떤 특성이 모델의 예측에 많이 관여했는지 한눈에 파악하는 데 유용하지만, 어디까지나 "모델이 이 특성을 많이 활용했다"는 뜻이지 "이 특성이 결과의 실제 원인이다"라는 인과관계를 말해주는 건 아니라는 점은 분명히 구분해야 한다. 상관관계가 높은 특성들이 있으면 중요도가 그 특성들 사이에 나뉘어 실제 영향력보다 낮게 잡히는 경우도 있다.

### 6. 부스팅과 AdaBoost

부스팅은 배깅과는 접근 자체가 다르다. 배깅이 여러 모델을 각자 독립적으로 학습시켜 결과를 모으는 방식이라면, 부스팅은 **이전 모델이 잘 못 맞힌 부분을 보완하도록 다음 모델을 순차적으로 추가**해나가는 방식이다. 앞 단계의 결과가 다음 단계의 학습 방향에 직접 영향을 준다는 점이 핵심 차이다.

**AdaBoost(Adaptive Boosting)**는 이 아이디어를 "잘못 분류된 샘플의 상대적 가중치를 높인다"는 방식으로 구현한다.

학습 과정을 단계별로 보면:

1. 모든 훈련 샘플에 똑같은 가중치 $1/m$을 부여한다.
2. 그 가중치를 반영해서 약한 분류기(보통 깊이가 얕은 트리)를 학습시킨다.
3. 이 분류기의 **가중 오류율** $r_j$를 계산한다. 그리고 이 오류율을 바탕으로 분류기의 최종 투표 기여도(가중치) $\alpha_j$를 정한다:

$$\alpha_j = \eta \log\frac{1 - r_j}{r_j}$$

여기서 $\eta$는 학습률이다. 오류율이 낮을수록(잘 맞힐수록) $\alpha_j$가 커져서 최종 투표에서 발언권이 세지고, 오류율이 0.5(무작위 추측 수준)에 가까울수록 $\alpha_j$는 0에 가까워진다.

4. 이 분류기가 틀린 샘플들의 가중치를 $\alpha_j$에 비례해서 끌어올리고, 전체 가중치 합이 1이 되도록 다시 정규화한다.
5. 갱신된 가중치를 가지고 다음 분류기를 학습시키고, 이 과정을 정해진 횟수만큼 반복한다.
6. 최종 예측은 각 분류기의 예측을 그 분류기의 가중치 $\alpha_j$로 가중 투표해서 결정한다.

이 과정을 거치면서, 뒤에 학습되는 분류기일수록 앞선 분류기들이 계속 헷갈려 하던(가중치가 커진) 샘플에 더 집중하게 된다. 기본 학습기로는 깊이가 딱 1인 결정 트리, 즉 뿌리에서 리프까지 조건 하나만 확인하는 **결정 스텀프(decision stump)**를 자주 쓴다. 개별 스텀프는 아주 약한 분류기지만, 이걸 여러 개 순차적으로 결합하면 상당히 복잡한 결정 경계도 학습할 수 있다는 게 부스팅의 묘미다.

`n_estimators`는 학습기 개수를, `learning_rate`는 각 학습기의 가중치 갱신 강도(위 식의 $\eta$)를 조절한다. 과적합 징후가 보이면 학습기 수를 줄이거나, 기반 모델에 더 강한 규제를 거는 방향으로 조정한다.

AdaBoost는 단순한 모델들을 결합해 복잡한 경계까지 학습할 수 있다는 장점이 있지만, 단점도 뚜렷하다. 잘못 분류된 샘플의 가중치를 계속 높여가는 구조이다 보니, 그 샘플이 사실은 잡음이나 잘못 붙은 레이블이었을 경우에도 알고리즘이 거기에 계속 집착하며 과적합할 위험이 있다. 또한 각 단계가 이전 단계의 결과에 의존하는 순차적 구조라서, 배깅처럼 모델들을 동시에 병렬로 학습시킬 수 없다는 것도 실무적인 제약이다.

### 7. 그레이디언트 부스팅

그레이디언트 부스팅도 부스팅 계열이라 모델을 순차적으로 추가한다는 점은 AdaBoost와 같지만, 접근 방식이 다르다. AdaBoost처럼 샘플의 가중치를 조정하는 대신, **현재까지 만들어진 앙상블이 남긴 오차 자체를 다음 모델이 직접 예측하도록 학습시킨다.**

책의 제곱오차 회귀 예제를 기준으로 풀어보면:

1. 첫 번째 트리를 훈련 데이터에 대해 학습시킨다.
2. 실제값에서 지금까지의 예측값을 뺀 **잔차(residual)**를 계산한다.
3. 두 번째 트리는 원래 타깃값이 아니라 이 잔차를 예측하도록 학습시킨다.
4. 새 트리의 예측값을 기존 앙상블의 예측에 (학습률만큼 축소해서) 더한다.
5. 여전히 남아 있는 오차를 대상으로 3~4를 반복한다.

이 과정을 수식으로 쓰면:

$$F_t(x) = F_{t-1}(x) + \eta \, h_t(x)$$

$F_{t-1}$은 지금까지 쌓인 앙상블, $h_t$는 새로 추가하는 모델, $\eta$는 학습률이다. 제곱오차 손실에서는 이 "잔차"가 정확히 손실 함수를 모델 예측값에 대해 미분한 것의 음의 방향(negative gradient)과 일치한다 — 그래서 이름이 그레이디언트 부스팅이다. 로그 손실 같은 다른 손실 함수를 쓸 때는 잔차라는 표현 대신, 일반적으로 그 손실 함수의 음의 기울기를 새 모델이 근사하도록 학습시킨다고 이해하면 된다.

**학습률과 트리 수는 항상 함께 조절해야 하는 한 쌍이다.** 학습률을 낮추면 트리 하나하나가 앙상블 전체에 기여하는 몫이 작아지므로, 같은 성능에 도달하려면 그만큼 더 많은 트리가 필요해진다. 대신 한 번에 크게 보정하지 않고 아주 조금씩 여러 번에 걸쳐 보정해나가기 때문에 일반화 성능에는 오히려 도움이 되는 경우가 많다. 이렇게 학습률을 낮춰 조금씩 기여하게 만드는 전략을 **축소(shrinkage)**라고 부른다.

트리 수가 너무 적으면 아직 오차를 충분히 줄이지 못한 과소적합 상태고, 너무 많으면 결국 훈련 데이터의 잡음까지 하나하나 쫓아가며 과적합하게 된다. 이걸 자동으로 조절하는 방법이 **조기 종료(early stopping)**다. 훈련 데이터 일부를 검증용으로 떼어두고, 검증 성능이 일정 기간(`n_iter_no_change`) 동안 `tol` 이상 개선되지 않으면 트리 추가를 멈춘다. `validation_fraction`으로 검증에 쓸 비율을 정한다.

또 `subsample`을 1보다 작게 설정하면, 매 트리마다 전체 훈련 데이터가 아니라 무작위로 뽑은 일부만 써서 학습한다. 이를 **확률적 그레이디언트 부스팅(Stochastic Gradient Boosting)**이라 부르며, 훈련 속도를 높이는 동시에 배깅과 비슷한 원리로 분산을 줄이는 효과도 있다.

### 8. 히스토그램 기반 그레이디언트 부스팅

**히스토그램 기반 그레이디언트 부스팅(HGB)**은 연속값 특성을 처리하는 방식을 바꿔서 속도를 크게 높인 변형이다. 일반적인 그레이디언트 부스팅은 노드를 분할할 때 그 특성이 가질 수 있는 모든 연속값 후보를 다 검토하지만, HGB는 미리 각 특성값을 몇 개의 구간(bin)으로 나눠두고 그 구간 경계만 분할 후보로 검토한다. 이렇게 하면 대용량 데이터에서도 계산량과 메모리 사용량을 크게 줄일 수 있다.

주요 하이퍼파라미터는 구간 수를 정하는 `max_bins`, 부스팅 반복(트리 추가) 횟수를 정하는 `max_iter`, 그리고 개별 트리의 복잡도를 제한하는 `max_leaf_nodes`, `min_samples_leaf` 등이다.

결측값과 범주형 특성을 별도의 전처리 없이 바로 다룰 수 있다는 것도 실용적인 장점이다. 다만 범주형 특성을 쓸 때는 해당 특성이 범주형이라는 사실을 모델에게 명시적으로 알려줘야 제대로 인식한다는 점은 유의해야 한다.

구간화는 계산을 빠르게 해주지만, 그 대가로 연속값이 가진 세밀한 수치 정보 일부를 잃는다. 이 정보 손실이 마치 규제처럼 작용해서 과적합을 억제하는 데 오히려 도움이 되기도 하지만, 구간 수를 너무 적게 잡으면 반대로 필요한 패턴조차 구분하지 못하는 과소적합으로 이어질 수 있다. 책에서는 같은 아이디어를 훨씬 더 정교하게 구현한 관련 라이브러리로 **XGBoost, LightGBM, CatBoost** 등도 함께 소개한다 — 셋 다 히스토그램 기반 최적화, 병렬·GPU 학습, 정규화 항 추가 등 실무에서 자주 쓰이는 다양한 개선을 얹은 구현체들이다.

### 9. 스태킹

**스태킹(Stacking)**은 여러 모델의 예측을 단순히 투표하거나 평균 내는 대신, **그 예측들을 어떻게 결합할지를 또 다른 모델이 학습하도록** 하는 방식이다. 이 최종 결합 모델을 **메타 학습기(meta learner)** 또는 **블렌더(blender)**라고 부른다.

예를 들어 서로 다른 세 모델이 각각 예측값을 내놓는다면, 이 세 예측값을 새로운 세 개의 특성으로 취급하고, 메타 모델이 이 세 값을 입력받아 최종 예측을 만들어낸다. 단순 평균이 "모든 모델을 똑같은 비중으로 믿는다"는 가정이라면, 스태킹은 "어떤 상황에서는 이 모델을, 다른 상황에서는 저 모델을 더 믿어야 한다"는 걸 메타 모델이 데이터로부터 스스로 배우게 하는 셈이다.

여기서 가장 중요한 건 메타 모델을 학습시킬 데이터를 어떻게 만드느냐다. 만약 기반 모델들이 이미 학습에 썼던 바로 그 샘플에 대한 예측값을 그대로 메타 모델의 입력으로 쓴다면, 기반 모델은 이미 그 정답을 "외운" 상태이기 때문에 실제 일반화 능력보다 지나치게 좋은 예측값이 메타 모델에 전달된다. 그러면 메타 모델은 이 과신된 신호를 믿고 학습하게 되어 전체 시스템이 심하게 과적합한다.

그래서 실제로는 교차검증을 이용해 **각 샘플에 대해, 그 샘플을 훈련에 쓰지 않은 폴드(fold)의 모델이 만든 예측값**, 즉 out-of-fold 예측을 만들어낸다. 이 방식은 4절에서 다룬 OOB 예측과 발상이 비슷하다 — 둘 다 "그 샘플을 안 본 모델의 예측을 써야 진짜 일반화 성능에 가깝다"는 원칙을 공유한다. 이렇게 만든 out-of-fold 예측들로 메타 모델을 학습시키고 나면, 기반 모델들은 그제서야 전체 훈련 데이터로 다시 한 번 학습시켜 실전 배포에 쓴다.

스태킹은 서로 다른 성격의 모델들(예: 선형 모델 + 트리 기반 모델 + 신경망)이 가진 각자의 강점을 메타 모델이 상황에 맞게 조합해 활용할 수 있다는 장점이 있다. 다만 교차검증 과정과 추가로 학습시켜야 하는 메타 모델 때문에 전체 계산량과 시스템 복잡도가 눈에 띄게 늘어난다. 메타 학습기 위에 또 다른 메타 학습기를 쌓는 식으로 여러 층으로 확장할 수도 있지만, 층을 늘릴수록 얻는 성능 향상이 그만큼 늘어난 계산 비용과 복잡도를 정당화하는지는 매번 따져봐야 한다.

## Chapter 7. 차원 축소 (Dimensionality Reduction)

### 1. 차원 축소의 필요성과 차원의 저주

여기서 말하는 차원이란 데이터가 가진 특성의 개수다. 특성이 많아지면 그만큼 표현할 수 있는 정보량도 늘어나지만, 그게 항상 학습에 유리하게 작용하는 건 아니다.

차원이 높아질수록 같은 수의 샘플이라도 공간 안에서 점점 더 듬성듬성 흩어지게 된다. 직관적인 예로, 단위 정사각형(2차원) 안에 균일하게 흩어진 점들과 단위 초입방체(고차원) 안에 흩어진 같은 수의 점들을 비교하면, 차원이 올라갈수록 임의의 점이 초입방체의 "경계" 근처에 있을 확률이 급격히 높아진다. 그 결과 새로운 샘플이 기존 훈련 샘플들로부터 멀리 떨어져 있을 가능성도 커지고, 거리나 유사도에 기반해 예측하는 모델(예: k-최근접 이웃)은 "가까운 이웃"이라는 개념 자체가 흐려지면서 성능이 떨어지기 쉽다. 계산량이 늘어나는 건 물론이고, 특성이 늘어난 만큼 잡음에 우연히 맞춰 학습할 위험도 커진다. 이런 일련의 문제를 통틀어 **차원의 저주**라고 부른다.

낮은 차원에서와 같은 수준의 데이터 밀도를 고차원에서도 유지하려면, 차원이 하나 늘어날 때마다 필요한 샘플 수가 기하급수적으로 늘어난다. 그래서 "데이터를 좀 더 모으면 되지 않을까"라는 직관적인 해법이 고차원에서는 현실적으로 통하지 않는 경우가 많다.

이런 맥락에서 차원 축소는 중복되거나 예측에 별 도움이 안 되는 정보를 걸러내어 학습 속도를 높이고 저장 공간을 절약하는 데 쓰인다. 그리고 데이터를 2차원이나 3차원으로 줄이면 사람이 눈으로 볼 수 있는 형태로 시각화할 수 있다는 것도 실무에서 자주 쓰이는 용도다.

다만 **차원 축소가 항상 예측 성능을 높여주는 건 아니라는 점**을 분명히 해둘 필요가 있다. 차원을 줄이는 과정에서 예측에 꼭 필요했던 정보까지 함께 사라질 수 있고, 축소 자체에도 계산 비용이 든다. 그래서 일단 원본 특성 그대로 학습해보고, 훈련이 너무 느리거나 차원의 저주 징후가 뚜렷할 때 차원 축소를 시도해보는 순서가 합리적이다.

### 2. 투영과 매니폴드 학습

차원 축소를 접근하는 방식은 크게 두 갈래로 나뉜다.

**투영(Projection)**은 데이터를 더 낮은 차원의 부분공간으로 그대로 옮기는 방식이다. 예를 들어 3차원 공간에 흩어진 데이터가 실제로는 거의 하나의 평면 근처에 모여 있다면, 그 평면에 수직으로 투영해서 2차원 좌표로 표현할 수 있다. PCA와 랜덤 투영이 이 계열의 대표적인 방법이다.

**매니폴드 학습**은 접근이 조금 다르다. 데이터가 고차원 공간 안에 놓여 있긴 하지만, 실제로는 그 공간 속에서 낮은 차원의 휘어진 구조(매니폴드)를 따라 분포하고 있다고 가정한다. 책에서 자주 드는 예시인 **Swiss roll**은, 원래는 2차원 평면인 종이를 돌돌 말아서 3차원 공간 안에 밀어 넣은 형태로 이해하면 된다.

이 Swiss roll을 그냥 단순하게 어떤 평면(예: $xy$ 평면)에 수직으로 투영해버리면, 원래는 종이 위에서 멀리 떨어져 있던 서로 다른 "층"들이 투영 후에는 겹쳐서 뒤섞여 버린다. 반면 말려 있는 구조를 인식하고 그 구조를 따라 조심스럽게 펼쳐내면, 원래 종이 위에서 가까웠던 점들이 펼친 뒤에도 여전히 가깝게 유지되는, 즉 원래의 이웃 관계를 훨씬 잘 보존하는 2차원 표현을 얻을 수 있다.

이처럼 "실제 데이터는 고차원 공간 안에서 무작위로 퍼져 있는 게 아니라 저차원 구조 근처에 놓여 있을 것"이라는 가정을 **매니폴드 가정**이라 부른다. 다만 여기서 한 가지 착각하기 쉬운 부분이 있는데, 데이터를 매니폴드를 따라 잘 펼쳐냈다고 해서 그 안의 분류 경계까지 자동으로 단순해지는 건 아니다. 매니폴드 위에서도 클래스 경계가 여전히 복잡하게 꼬여 있을 수 있으며, 오히려 원래 고차원에서는 초평면 하나로 간단히 나뉘던 경계가 펼친 뒤에는 더 복잡해지는 경우도 있다.

### 3. PCA의 핵심 원리

**PCA(Principal Component Analysis, 주성분 분석)**는 데이터를 투영할 방향을 고를 때, **분산을 최대한 많이 보존하는 방향**을 우선적으로 선택하는 방법이다.

왜 분산을 기준으로 삼는지 생각해보면, 분산이 큰 방향으로 투영할수록 데이터 포인트들이 그 축 위에서 서로 잘 구분되어 퍼져 있게 되고, 반대로 분산이 거의 없는 방향으로 투영하면 모든 점이 한 점 근처로 뭉개져서 정보를 거의 다 잃게 된다. 그래서 "가능한 한 정보 손실을 줄이면서 차원을 줄이자"는 목표를 "분산을 최대한 보존하는 축을 찾자"는 문제로 바꿔서 푸는 것이다.

첫 번째 주성분은 데이터가 가장 크게 퍼져 있는 방향이다. 두 번째 주성분은 첫 번째 주성분과 직교(수직)하면서, 남은 분산을 가장 많이 설명하는 방향으로 정해진다. 세 번째 이후도 마찬가지로, 이전 주성분들과 모두 직교하면서 남은 분산을 최대한 설명하는 방향을 차례로 찾아나간다.

여기서 중요한 건, 주성분이 원래 있던 특성 중 하나를 그대로 고르는 게 아니라, 원래 특성들을 어떤 비율로 섞어서 만든(선형 결합한) **완전히 새로운 축**이라는 점이다. 예를 들어 "키"와 "몸무게"라는 두 특성이 있었다면, 첫 번째 주성분은 "키와 몸무게 각각에 어떤 계수를 곱해 더한 값" 같은 새로운 합성 변수가 된다.

PCA의 학습 흐름을 정리하면:

1. 각 특성에서 그 특성의 평균을 빼서, 데이터의 중심을 원점으로 옮긴다.
2. 이렇게 중심화된 데이터 행렬에 **특잇값 분해(SVD)**를 적용한다. (수학적으로는 데이터의 공분산 행렬을 고유값 분해하는 것과 동치인 결과를 얻는데, SVD를 직접 쓰면 공분산 행렬을 명시적으로 계산하지 않아도 돼서 수치적으로 더 안정적이다.)
3. SVD 결과에서 나오는 특이값의 크기 순서대로, 즉 분산을 많이 설명하는 순서대로 주성분(축)을 정렬해 선택한다.
4. 선택한 축들에 원본 데이터를 투영해서 저차원 표현을 얻는다.

PCA는 주어진 목표 차원 수 안에서 만들 수 있는 모든 선형 투영 중 분산을 가장 많이 보존하는 투영이며, 이는 곧 저차원으로 투영했다가 다시 원래 차원으로 복원했을 때 생기는 **제곱 재구성 오차를 최소화**하는 것과 수학적으로 동치인 목표라는 점도 알아두면 좋다. 즉 "분산을 최대한 보존한다"와 "복원했을 때 원본과의 오차를 최소화한다"는 같은 문제를 서로 다른 각도에서 본 것이다.

사이킷런의 `PCA`는 위 1번 단계인 중심화(평균 빼기)는 자동으로 처리해준다. 다만 **특성 단위를 맞추는 표준화까지 자동으로 해주지는 않는다.** 예를 들어 한 특성은 단위가 "미터"이고 다른 특성은 "킬로그램"이어서 값의 스케일 자체가 크게 다르다면, 스케일이 큰 특성 쪽이 실제 중요도와 무관하게 분산도 더 커 보여서 주성분 방향을 왜곡시킬 수 있다. 그래서 특성들의 단위가 크게 다른 데이터를 다룰 때는 PCA를 적용하기 전에 `StandardScaler` 등으로 스케일링이 필요한지 반드시 검토해야 한다.

또 하나 명심할 점은, PCA는 정답 레이블을 전혀 참고하지 않는 **비지도 학습**이라는 것이다. 분산이 가장 큰 방향이 반드시 클래스를 잘 구분해주는 방향이라는 보장은 없다. 실제로는 분산이 작아서 PCA가 무시해버린 방향에 분류에 결정적인 정보가 숨어 있는 경우도 있을 수 있다. 그래서 PCA를 전처리로 쓸 때는 반드시 후속 모델의 성능으로 그 선택이 적절했는지 검증해봐야 한다(레이블을 활용하는 지도 방식 축소인 LDA는 뒤에서 다시 다룬다).

### 4. 보존할 차원 수와 설명된 분산

각 주성분이 전체 분산 중 어느 정도 비중을 설명하는지를 **설명된 분산 비율**이라 하고, 사이킷런에서는 `explained_variance_ratio_` 속성으로 확인할 수 있다.

몇 차원까지 남길지 정하는 방법은 크게 다음과 같다.

- 주성분들의 설명된 분산 비율을 누적해서, 예를 들어 95% 같은 목표 수준에 도달할 때까지 필요한 최소 개수를 선택한다.
- 누적 설명 분산을 차원 수에 대해 그린 곡선(스크리 플롯)을 보고, 곡선의 기울기가 급격히 완만해지는 "팔꿈치" 지점을 참고해서 그 이후는 큰 의미가 없다고 판단한다.
- 차원 축소가 전처리 목적이라면, 축소 후 데이터로 실제 분류·회귀 모델을 학습시켜 교차검증 성능이 가장 좋은 차원 수를 그리드 서치로 찾는다.
- 목적이 시각화라면 애초에 사람이 볼 수 있는 2차원이나 3차원으로 정한다.

한 가지 꼭 짚어야 할 오해가 있다. `n_components=0.95`라는 설정은 "원래 특성 784개 중 95%인 약 745개 특성을 남긴다"는 뜻이 절대 아니다. **전체 분산의 95% 이상을 설명하는 데 필요한 최소한의 주성분 개수를 자동으로 선택한다**는 뜻이다. 예를 들어 MNIST 손글씨 데이터(784차원)의 경우, 책에서는 분산의 95%를 보존하는 데 필요한 주성분이 대략 150여 개 수준으로 나온다고 소개한다 — 즉 784차원을 150차원 정도로 줄여도 원래 정보의 95%가량은 유지된다는 뜻이다. 그리고 이 "95%"라는 숫자는 뒤에 이어지는 분류 모델의 정확도가 95%가 된다는 뜻도 전혀 아니라는 점 역시 헷갈리지 말아야 한다 — 어디까지나 분산 보존 비율일 뿐, 예측 정확도와는 직접적인 관계가 없다.

필요한 차원 수는 결국 데이터 자체의 구조에 달려 있다. 특성들 사이에 중복(상관관계)이 크다면 적은 수의 주성분만으로도 원래 분산의 대부분을 설명할 수 있고, 반대로 특성들이 서로 거의 독립적이라면 분산을 웬만큼 보존하려 해도 원래 차원 수와 크게 다르지 않은 만큼의 주성분이 필요할 수 있다.

### 5. PCA를 이용한 압축과 복원

차원을 줄이면 샘플 하나를 표현하는 데 필요한 숫자의 개수가 줄어드는 만큼, 데이터를 압축하는 효과가 자연스럽게 생긴다.

`inverse_transform()` 메서드를 쓰면 축소된 저차원 데이터를 다시 원래의 고차원 특성 공간으로 되돌릴 수 있다. 하지만 애초에 버린 주성분들이 담고 있던 정보는 이미 사라졌기 때문에, 복원된 데이터가 원본과 완전히 똑같아지는 경우는 (모든 주성분을 다 남기지 않는 한) 거의 없다.

원본 데이터와 복원된 데이터 사이의 차이를 측정한 것이 **재구성 오차(reconstruction error)**다. 일반적으로 보존하는 차원 수가 많을수록 재구성 오차는 줄어들지만, 그만큼 압축 효과(데이터 크기 절감)는 줄어드는 트레이드오프 관계에 있다.

다만 재구성 오차 하나만으로 차원 축소의 가치를 전부 판단할 수는 없다는 점도 짚어야 한다. 만약 진짜 목적이 압축이 아니라 후속 분류 모델의 전처리라면, 재구성 오차가 작다고 해서 그게 곧 좋은 분류 성능을 보장하지는 않는다. 이 경우엔 반드시 축소된 데이터로 학습시킨 분류기의 실제 성능까지 함께 확인해봐야 한다.

### 6. 무작위 PCA와 점진적 PCA

일반적인 (완전) PCA는 SVD를 계산해야 하는데, 이 계산량이 데이터 크기와 특성 수가 커질수록 빠르게 늘어나서 대용량 데이터에서는 계산량과 메모리 부담이 커진다. 이를 보완하는 두 가지 방법이 있다.

**무작위 PCA(Randomized PCA)**는 전체 주성분을 정밀하게 계산하는 대신, 확률적 근사 알고리즘을 이용해 우리가 실제로 필요로 하는 상위 몇 개의 주성분만 훨씬 빠르게 근사해서 찾아낸다. 원래 차원 수에 비해 남기고 싶은 차원 수가 훨씬 작을 때 특히 속도 이점이 크다. 사이킷런에서는 `PCA(svd_solver="randomized")`로 쓸 수 있다.

**점진적 PCA(Incremental PCA)**는 접근 자체가 다른데, 전체 데이터를 한 번에 메모리에 올리는 대신 데이터를 작은 배치(미니배치)로 나눠서 순차적으로 처리한다. 전체 데이터셋이 메모리 용량을 초과할 정도로 클 때 특히 유용하며, `partial_fit()` 메서드를 배치마다 호출해서 조금씩 학습시켜 나간다. 책에서는 디스크에 저장된 대용량 데이터를 필요한 부분만 그때그때 읽어오는 메모리 매핑(`np.memmap`) 방식을 점진적 PCA와 함께 활용하는 방법도 소개한다.

점진적 PCA는 메모리 부담을 크게 줄여주지만, 데이터를 한 번에 전체로 보고 계산하는 게 아니라 배치 단위로 근사적으로 갱신해나가는 방식이기 때문에, 전체 데이터를 한 번에 처리한 일반 PCA와 결과가 완전히 똑같지는 않을 수 있다는 점을 감안해야 한다.

한 가지 실무적으로 중요한 주의사항이 있는데, PCA를 다시(새로운 데이터로) 학습시키면 주성분의 방향이나 순서가 이전과 달라질 수 있다. 그래서 PCA를 갱신할 때는 PCA의 출력을 입력으로 쓰고 있던 후속 모델(분류기 등)도 반드시 함께 다시 학습시켜야 한다는 점을 잊지 말아야 한다.

### 7. 랜덤 투영

**랜덤 투영(Random Projection)**은 앞선 방법들과 접근이 근본적으로 다르다. 데이터에서 주성분을 찾는 과정 자체가 없고, 대신 **무작위로 생성한 투영 행렬**을 데이터에 곱해서 낮은 차원으로 옮긴다.

이게 말이 되나 싶을 수 있는데, 이론적 근거가 있다. **존슨–린덴스트라우스 보조정리(Johnson–Lindenstrauss lemma)**에 따르면, 충분히 큰 출력 차원만 확보한다면 고차원 공간의 점들을 무작위 방향으로 투영하더라도 점들 사이의 거리가 높은 확률로 근사적으로 보존된다는 것이 수학적으로 증명되어 있다. 즉 "무작위로 방향을 골라도, 개수만 충분하면 원래의 거리 구조가 크게 망가지지 않는다"는 다소 반직관적인 결과다. 다만 이건 어디까지나 "충분한 차원"을 전제로 한 확률적 보장이고, 목표 차원을 너무 공격적으로 줄이면 그만큼 거리 왜곡이 커진다.

목표 차원은 데이터의 샘플 수와 허용하는 거리 왜곡 수준에 따라 달라진다. 허용 오차를 작게 잡을수록(더 정확한 거리 보존을 요구할수록) 더 많은 차원이 필요해진다.

사이킷런은 두 가지 구현을 제공한다.

- `GaussianRandomProjection`: 각 원소를 가우스 분포에서 뽑아 만든 투영 행렬을 쓴다.
- `SparseRandomProjection`: 대부분의 원소가 0인 희소 행렬을 투영 행렬로 쓴다. 메모리 사용량과 연산량을 크게 줄이면서도 비슷한 거리 보존 성질을 얻을 수 있다.

주요 파라미터로는 출력 차원을 직접 지정하는 `n_components`, 목표 차원을 자동으로 계산할 때 허용할 거리 왜곡 수준을 정하는 `eps`, 그리고 희소 투영 행렬에서 0이 아닌 원소의 비율을 조절하는 `density`가 있다.

랜덤 투영은 데이터에서 중요한 축을 찾는 최적화 과정이 아예 없기 때문에 계산이 매우 빠르며, 특성 수가 극단적으로 많거나 희소한 데이터에서 특히 유용하다. 반면 데이터의 실제 구조에 맞춰 방향을 고르는 게 아니라 순전히 무작위이므로, PCA에 비해 정말 중요한 신호까지 함께 손실될 가능성이 더 높다. 의사역행렬(pseudo-inverse)을 이용해 원래 차원으로 근사 복원을 시도할 수는 있지만, PCA만큼 깔끔한 복원은 기대하기 어렵고 역변환 계산 자체의 비용도 작지 않을 수 있다.

여기서 자주 헷갈리는 두 개념을 확실히 구분해두면 좋다. **무작위 PCA는 여전히 데이터를 보고 주성분을 (근사적으로) 찾아내는 방법**이고, **랜덤 투영은 데이터를 아예 보지 않고 주성분을 찾는 과정 없이 무작위 방향으로 투영하는 방법**이다. "무작위"라는 단어가 둘 다 들어가서 헷갈리기 쉽지만, 전자는 계산 방식만 근사적(무작위성을 이용한 알고리즘)이고 후자는 투영 방향 자체가 데이터와 무관하게 무작위라는 점에서 근본적으로 다르다.

### 8. LLE

**LLE(Locally Linear Embedding)**는 비선형 차원 축소 방법으로, 매니폴드 학습 계열에 속한다. 목표는 **각 샘플과 그 근처 이웃들 사이의 국소적인(local) 관계를 저차원에서도 그대로 유지하는 것**이다.

학습은 두 단계로 나눠서 이해하면 명확하다.

**1단계**: 각 샘플에 대해 최근접 이웃들을 찾은 뒤, 그 샘플을 이웃들의 가중합으로 최대한 잘 재구성할 수 있는 가중치를 구한다. 즉 "이 샘플은 이웃 A를 30%, 이웃 B를 50%, 이웃 C를 20% 섞으면 거의 그대로 재현된다"는 식의 국소적인 선형 관계를 데이터로부터 계산해내는 것이다. 이때 각 샘플에 대한 가중치의 합이 1이 되도록 제약을 둔다.

**2단계**: 1단계에서 구한 가중치는 그대로 고정해둔 채, 이번엔 반대로 "이 가중치 조합을 낮은 차원에서도 그대로 유지했을 때 가장 잘 재구성되는 좌표"를 찾는다. 다시 말해, 원래 고차원 공간에서 성립했던 국소적인 선형 관계(누가 누구를 얼마나 섞은 것인지)를 저차원 공간에서도 동일하게 유지하도록 좌표를 배치하는 것이다.

주요 하이퍼파라미터는 출력 차원인 `n_components`와 이웃 개수인 `n_neighbors`다. 이웃 수를 너무 작게 잡으면 국소 구조를 추정할 근거가 부족해져 결과가 불안정해질 수 있고, 반대로 너무 크게 잡으면 원래는 매니폴드 위에서 서로 멀리 떨어져 있어야 할 부분(예: Swiss roll에서 말려 있는 서로 다른 층)까지 "이웃"으로 잘못 연결해버려서 오히려 국소 관계를 왜곡시킬 수 있다.

LLE는 Swiss roll처럼 뚜렷하게 휘어진 구조를 펼쳐내는 데 특히 효과적이다. 다만 잡음에 상대적으로 민감하고, "국소적인" 이웃 관계는 잘 보존하지만 멀리 떨어진 두 샘플 사이의 전역적인 거리까지 정확하게 보존해주는 건 아니라는 한계가 있다. 또 모든 샘플 쌍에 대한 이웃 탐색과 최적화가 필요해서, 데이터가 커질수록 계산 부담도 함께 커진다.

### 9. 다른 차원 축소 기법

이 밖에도 책에서 간단히 소개하는 방법들이 있다.

- **MDS(Multidimensional Scaling)**: 고차원에서 측정한 샘플 사이의 거리를 가능한 한 그대로 유지하면서 저차원에 배치하는 방법이다.
- **Isomap**: 이웃 그래프를 만들고 그 그래프 위에서의 최단 경로 거리로 매니폴드를 따라가는 "곡면 거리"를 근사한 뒤, 그 거리를 보존하도록 차원을 줄인다. 직선 거리 대신 곡면을 따라가는 거리를 쓴다는 점에서 단순 MDS와 구분된다.
- **t-SNE**: 가까이 있는 샘플들의 유사성을 특히 강조해서 표현하는 방법으로, 주로 고차원 데이터를 2차원이나 3차원으로 시각화하는 용도로 쓰인다. 군집 구조를 시각적으로 뚜렷하게 드러내는 데는 강력하지만, 축의 절대적인 위치나 클러스터 간 거리가 실제 데이터의 전역 구조를 정확히 반영하지는 않을 수 있어서, 일반적인 예측 모델의 전처리(feature engineering) 목적과는 쓰임새를 구분해서 이해해야 한다.
- **LDA(Linear Discriminant Analysis)**: 클래스 레이블을 적극적으로 활용해서, 같은 클래스끼리는 가깝게 모이고 서로 다른 클래스는 멀리 떨어지도록 만드는 방향을 찾는다. PCA가 레이블 없이 분산만 보고 방향을 정하는 비지도 방식이라면, LDA는 레이블을 직접 이용하는 **지도 방식의 차원 축소**라는 점에서 근본적으로 다르다.
- **UMAP(Uniform Manifold Approximation and Projection)**: 국소 구조뿐 아니라 전역 구조까지 함께 고려하려고 시도하는 비교적 최신 기법으로 소개된다. 사이킷런에 기본 내장되어 있지 않아서, 책에서는 별도의 `umap-learn` 패키지를 설치해서 쓰는 방법을 안내한다.

결국 어떤 차원 축소 방법을 고를지는 "그림이 얼마나 예쁘게 나오는가"가 아니라 **무엇을 보존해야 하는가**에 달려 있다. 목적이 압축이라면 재구성 오차와 실제 저장 용량 절감분을, 목적이 예측 모델의 전처리라면 축소된 데이터로 학습시킨 후속 모델의 실제 성능을, 목적이 시각화라면 이웃 관계나 전역 구조가 얼마나 왜곡되는지를 각각 기준으로 삼아 판단해야 한다.